# Baseline strategies and live trading monitor

Запуск `RelativeMomentumBaseline` или `RollingHedgeBaseline` с обновляемыми графиками, метриками и лентой сделок. Тот же монитор принимает будущую PPO/SAC-модель через `ModelStrategy`.

In [ ]:
from pathlib import Path
import sys

try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT = Path('/content/drive/MyDrive/gas_portfolio_rl')
except ImportError:
    IN_COLAB = False
    cwd = Path.cwd().resolve()
    PROJECT_ROOT = cwd if (cwd / 'pyproject.toml').exists() else cwd / 'gas_portfolio_rl'

# Подключаем bootstrap из корня, затем он всегда ставит PROJECT_ROOT/src первым в sys.path.
sys.path.insert(0, str(PROJECT_ROOT))
from notebook_bootstrap import bootstrap_project
PROJECT_ROOT = bootstrap_project(PROJECT_ROOT, install_dependencies=IN_COLAB)

import gas_portfolio_rl
print('Runtime:', 'Google Colab' if IN_COLAB else 'local')
print('Project root:', PROJECT_ROOT)
print('Package imported from:', gas_portfolio_rl.__file__)

In [ ]:
from gas_portfolio_rl import AppConfig, GasPortfolioEnv
from gas_portfolio_rl.baselines import RelativeMomentumBaseline, RollingHedgeBaseline, SignedRollingHedgeBaseline, run_strategy
from gas_portfolio_rl.visualization import LiveTradingDisplay

config = AppConfig.from_yaml(
    PROJECT_ROOT / 'configs/market/default.yaml',
    PROJECT_ROOT / 'configs/environment/default.yaml',
    PROJECT_ROOT / 'configs/reward/default.yaml',
    PROJECT_ROOT / 'configs/baselines/default.yaml',
)

In [ ]:
STRATEGY = 'momentum'  # 'momentum', 'hedge' или 'signed_hedge'
SEED = 42
STEPS = 252
REFRESH_EVERY = 2
WINDOW_SIZE = 80
FRAME_PAUSE_SECONDS = 0.03

if STRATEGY == 'momentum':
    strategy = RelativeMomentumBaseline(
        config.baselines.relative_momentum,
        trading_days=config.market.trading_days,
    )
elif STRATEGY == 'hedge':
    strategy = RollingHedgeBaseline(config.baselines.rolling_hedge)
elif STRATEGY == 'signed_hedge':
    strategy = SignedRollingHedgeBaseline(config.baselines.signed_rolling_hedge)
else:
    raise ValueError(STRATEGY)

monitor = LiveTradingDisplay(
    refresh_every=REFRESH_EVERY,
    window_size=WINDOW_SIZE,
    pause_seconds=FRAME_PAUSE_SECONDS,
    live=True,
)
result = run_strategy(
    GasPortfolioEnv(config), strategy, seed=SEED, max_steps=STEPS, monitor=monitor
)

In [ ]:
output = PROJECT_ROOT / 'results/reports' / f'{result.strategy_name}_live_monitor.png'
monitor.save(output)
final = result.trajectory[-1]
{
    'strategy': result.strategy_name,
    'steps': final['step'],
    'portfolio_value': final['portfolio_value'],
    'pnl': final['pnl'],
    'total_reward': result.total_reward,
    'commissions': final['cumulative_transaction_cost'],
    'dividends': final['cumulative_dividends'],
    'margin_calls': final['margin_calls'],
    'saved_to': str(output),
}

## Подключение RL-модели

После обучения: `strategy = ModelStrategy(model, name='ppo')`, затем передайте её в тот же `run_strategy(...)`. Формат дисплея и ленты сделок не меняется.